[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmolSISL/IMDb-Deep-Learning-Research/blob/main/notebooks/04_RQ2_Transfer_Learning.ipynb)

# Notebook 04 — RQ2: Fine-Tuning and Cross-Domain Transfer

> **RQ2.** Does fine-tuning a pre-trained transformer outperform models trained from scratch, and how well does a model built on movie reviews transfer to product reviews as the amount of labeled target-domain data increases?

**Design**
1. **In-domain:** fine-tune DistilBERT (Sanh et al., 2019) on the IMDb training split and compare it with the RQ1 models on the official IMDb test set.
2. **Domain shift:** apply every IMDb model to the **Amazon Electronics test set (10,000)** without retraining. Accuracy drop = accuracy(IMDb test) − accuracy(Amazon test).
3. **Learning curves:** adapt with 250, 500, 1,000, 2,500 and 5,000 labeled Amazon reviews (5 seeds each), comparing three conditions:
   * **IMDb DistilBERT → Amazon** (transfer from the source domain);
   * **Generic DistilBERT → Amazon** (pre-trained language model only, no movie reviews), which separates the effect of pre-training from the effect of the source domain;
   * **TF-IDF + logistic regression trained from scratch** on the same Amazon labels (the RQ1 baseline family).
4. **Labels needed:** the smallest labeled set at which mean target accuracy is within **2 percentage points** of in-domain (IMDb) accuracy.

All models in this notebook use the **score-masked** text, following the mitigation in the interim report. Explicit scores such as "7/10" do not occur in product reviews, so a model must not rely on them.

⚙️ Requires a **GPU** runtime (T4 is sufficient). Runs are saved as they finish, so after a disconnect, re-running skips the completed runs.

## 0. Setup

In [ ]:
# ---------------------------------------------------------------
# Project setup: mount Google Drive so every notebook shares the same
# data, models and outputs (folder: MyDrive/customer-voice-intelligence)
# ---------------------------------------------------------------
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/customer-voice-intelligence"
else:  # running locally
    PROJECT_DIR = os.environ.get("CVI_PROJECT_DIR", os.path.abspath("customer-voice-intelligence"))

for sub in ["data/raw", "data/processed", "outputs/figures", "outputs/tables",
            "outputs/results", "models", "code"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
sys.path.insert(0, os.path.join(PROJECT_DIR, "code"))
print("Project folder:", PROJECT_DIR)

In [ ]:
import importlib, json, time, copy, math
import cvi_utils as U, cvi_models as M
importlib.reload(U); importlib.reload(M)
U.set_project(PROJECT_DIR)
import numpy as np, pandas as pd, joblib, torch
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
U.plot_style()
DEVICE = M.DEVICE
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Configuration

In [ ]:
CKPT = "distilbert-base-uncased"
MAX_LEN = 512                     # DistilBERT limit (subword tokens)
SRC = dict(lr=2e-5, batch_size=16, epochs=2, warmup=0.1, weight_decay=0.01)
SOURCE_SEEDS = [42]               # add seeds if GPU time allows, e.g. [42, 43, 44]
LC_SIZES = [250, 500, 1000, 2500, 5000]
LC_SEEDS = [42, 43, 44, 45, 46]
LC_EPOCHS = {250: 10, 500: 8, 1000: 5, 2500: 4, 5000: 3}   # more passes for tiny sets
LC = dict(lr=3e-5, batch_size=16, warmup=0.1, weight_decay=0.01)
LC_VAL_N = 1000                   # Amazon validation reviews used to pick the best epoch
RECOVERY_PP = 0.02
C_GRID = [0.5, 1, 2, 4, 8, 16, 32]
NAMES = {"logreg": "TF-IDF + LogReg", "ffn": "Feed-forward NN", "cnn": "1D-CNN", "bilstm": "BiLSTM", "distilbert": "DistilBERT"}

if U.QUICK:
    SRC["epochs"] = 1; LC_SIZES = [100, 300]; LC_SEEDS = [42, 43]; LC_EPOCHS = {100: 2, 300: 2}; LC_VAL_N = 100; MAX_LEN = 128

RES = U.P("outputs", "results"); MOD1 = U.P("models", "rq1"); MOD2 = U.P("models", "rq2"); os.makedirs(MOD2, exist_ok=True)
SHIFT_CSV, LC_CSV, SRC_CSV = f"{RES}/rq2_domain_shift.csv", f"{RES}/rq2_learning_curves.csv", f"{RES}/rq2_source_runs.csv"

## 2. Data

In [ ]:
imdb, amz = U.load_imdb(), U.load_amazon()
mask = lambda s: [U.mask_scores(t) for t in s]
T = {
    "imdb_train": imdb[imdb.split == "train"], "imdb_val": imdb[imdb.split == "validation"], "imdb_test": imdb[imdb.split == "test"],
    "amz_test": amz[amz.split == "test"], "amz_val": amz[amz.split == "validation"], "amz_pool": amz[amz.split == "finetune_pool"],
}
TXT = {k: mask(v.review_text) for k, v in T.items()}
Y = {k: v.label.to_numpy() for k, v in T.items()}
print({k: len(v) for k, v in T.items()})

tok = AutoTokenizer.from_pretrained(CKPT)
ENC = {k: tok(v, truncation=False, add_special_tokens=True)["input_ids"] for k, v in TXT.items()}
for k in ["imdb_train", "amz_test"]:
    lens = np.array([len(e) for e in ENC[k]])
    print(f"{k}: median {np.median(lens):.0f} subword tokens; {np.mean(lens > MAX_LEN):.1%} longer than {MAX_LEN} (truncated)")

def trunc(ids):     # keep [CLS] ... [SEP] within MAX_LEN
    return ids if len(ids) <= MAX_LEN else ids[:MAX_LEN - 1] + [ids[-1]]
ENC = {k: [trunc(e) for e in v] for k, v in ENC.items()}

## 3. DistilBERT training utilities
The utilities use dynamic padding, mixed precision on the GPU, AdamW with linear warm-up and decay, and keep the best epoch by validation accuracy.

In [ ]:
def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.zeros((len(seqs), L), dtype=torch.long); att = torch.zeros_like(ids)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
    return ids.to(DEVICE), att.to(DEVICE)

AMP = torch.cuda.is_available()

@torch.no_grad()
def bert_proba(model, enc, batch_size=64):
    model.eval()
    order = np.argsort([len(e) for e in enc])            # length bucketing = faster inference
    out = np.zeros(len(enc), dtype=np.float32)
    for i in range(0, len(enc), batch_size):
        idx = order[i:i + batch_size]
        ids, att = pad_batch([enc[j] for j in idx])
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
            logits = model(input_ids=ids, attention_mask=att).logits
        out[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
    return out

def bert_train(model, enc_tr, y_tr, enc_val, y_val, epochs, lr, batch_size, warmup, weight_decay, seed, verbose=True):
    U.set_seed(seed); model.to(DEVICE)
    no_decay = ["bias", "LayerNorm.weight"]
    groups = [{"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay},
              {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=lr)
    steps = epochs * math.ceil(len(enc_tr) / batch_size)
    sched = get_linear_schedule_with_warmup(opt, int(warmup * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    rng = np.random.RandomState(seed); y_tr = np.asarray(y_tr)
    best_acc, best_state, hist = -1, None, []
    t0 = time.perf_counter()
    for ep in range(1, epochs + 1):
        model.train(); perm = rng.permutation(len(enc_tr)); total = 0.0
        for i in range(0, len(perm), batch_size):
            idx = perm[i:i + batch_size]
            ids, att = pad_batch([enc_tr[j] for j in idx])
            labels = torch.tensor(y_tr[idx], dtype=torch.long, device=DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                loss = model(input_ids=ids, attention_mask=att, labels=labels).loss
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            total += loss.item() * len(idx)
        p = bert_proba(model, enc_val); acc = float(((p >= .5).astype(int) == np.asarray(y_val)).mean())
        hist.append(dict(epoch=ep, train_loss=total / len(perm), val_acc=acc, minutes=(time.perf_counter() - t0) / 60))
        if verbose:
            print(f"    epoch {ep}: train loss {total / len(perm):.4f}, val acc {acc:.4f}, {(time.perf_counter() - t0) / 60:.1f} min")
        if acc > best_acc:
            best_acc = acc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, hist, (time.perf_counter() - t0) / 60

def new_bert(path=CKPT):
    return AutoModelForSequenceClassification.from_pretrained(path, num_labels=2).to(DEVICE)

def done(csv):
    return set(pd.read_csv(csv).key) if os.path.exists(csv) else set()

## 4. Fine-tune DistilBERT on IMDb (source domain)

In [ ]:
for seed in SOURCE_SEEDS:
    key = f"distilbert_imdb_s{seed}"; path = f"{MOD2}/{key}"
    if key in done(SRC_CSV) and os.path.isdir(path):
        print("skip", key); continue
    print("Training", key)
    model, hist, minutes = bert_train(new_bert(), ENC["imdb_train"], Y["imdb_train"], ENC["imdb_val"], Y["imdb_val"],
                                      SRC["epochs"], SRC["lr"], SRC["batch_size"], SRC["warmup"], SRC["weight_decay"], seed)
    model.save_pretrained(path); tok.save_pretrained(path)
    p_imdb = bert_proba(model, ENC["imdb_test"]); p_amz = bert_proba(model, ENC["amz_test"])
    np.savez_compressed(f"{RES}/rq2_{key}_probs.npz", imdb_test=p_imdb, amz_test=p_amz)
    U.append_csv(dict(key=key, seed=seed, train_time_min=minutes, epochs=len(hist),
                      n_params=sum(p.numel() for p in model.parameters()),
                      imdb_test_acc=U.classification_metrics(Y["imdb_test"], p_imdb)["accuracy"],
                      amz_test_acc=U.classification_metrics(Y["amz_test"], p_amz)["accuracy"]), SRC_CSV)
    pd.DataFrame(hist).to_csv(f"{RES}/rq2_{key}_history.csv", index=False)
    del model; torch.cuda.empty_cache()
pd.read_csv(SRC_CSV)

## 5. Domain shift: IMDb models applied to Amazon without retraining
The RQ1 models trained on **score-masked** text (seed 42) and the IMDb DistilBERT are scored on both test sets.

In [ ]:
vec = joblib.load(f"{MOD1}/tfidf_masked.joblib"); vocab = json.load(open(f"{MOD1}/vocab_masked.json"))

def rq1_proba(kind, texts):
    if kind == "logreg":
        return joblib.load(f"{MOD1}/logreg_masked_s42.joblib").predict_proba(vec.transform(texts))[:, 1]
    spec = json.load(open(f"{MOD1}/{kind}_masked_s42.json"))
    m = M.make_model(kind, spec["cfg"], spec["n_in"], spec["vocab_size"])
    m.load_state_dict(torch.load(f"{MOD1}/{kind}_masked_s42.pt", map_location=DEVICE)); m.to(DEVICE)
    data = vec.transform(texts) if kind == "ffn" else U.encode(texts, vocab, 400)
    return M.predict_proba(m, data)

PROBS = {}
for kind in ["logreg", "ffn", "cnn", "bilstm"]:
    PROBS[kind] = {"imdb_test": rq1_proba(kind, TXT["imdb_test"]), "amz_test": rq1_proba(kind, TXT["amz_test"])}
src = np.load(f"{RES}/rq2_distilbert_imdb_s{SOURCE_SEEDS[0]}_probs.npz")
PROBS["distilbert"] = {"imdb_test": src["imdb_test"], "amz_test": src["amz_test"]}

rows = []
for k, p in PROBS.items():
    mi = U.classification_metrics(Y["imdb_test"], p["imdb_test"]); ma = U.classification_metrics(Y["amz_test"], p["amz_test"])
    rows.append(dict(model=k, imdb_acc=mi["accuracy"], amz_acc=ma["accuracy"], drop_pp=100 * (mi["accuracy"] - ma["accuracy"]),
                     imdb_macro_f1=mi["macro_f1"], amz_macro_f1=ma["macro_f1"], amz_roc_auc=ma["roc_auc"],
                     amz_recall_pos=ma["recall"], amz_recall_neg=ma["tn"] / (ma["tn"] + ma["fp"])))
shift = pd.DataFrame(rows); shift.to_csv(SHIFT_CSV, index=False)

table10 = pd.DataFrame({
    "Model": [NAMES[k] for k in shift.model],
    "IMDb test accuracy": (100 * shift.imdb_acc).map("{:.2f}%".format),
    "Amazon test accuracy (no retraining)": (100 * shift.amz_acc).map("{:.2f}%".format),
    "Accuracy drop (pp)": shift.drop_pp.map("{:.1f}".format),
    "Amazon macro F1": shift.amz_macro_f1.map("{:.4f}".format),
    "Amazon recall pos / neg": [f"{a:.2f} / {b:.2f}" for a, b in zip(shift.amz_recall_pos, shift.amz_recall_neg)],
})
U.save_table(table10, "table10_rq2_domain_shift")
table10

### Paired McNemar tests: DistilBERT versus the models trained from scratch

In [ ]:
t = []
for dom in ["imdb_test", "amz_test"]:
    pd_ = (PROBS["distilbert"][dom] >= .5).astype(int)
    for k in ["logreg", "ffn", "cnn", "bilstm"]:
        r = U.mcnemar(Y[dom], pd_, (PROBS[k][dom] >= .5).astype(int))
        t.append({"Test set": "IMDb" if dom == "imdb_test" else "Amazon", "Comparison": f"DistilBERT vs {NAMES[k]}",
                  "DistilBERT right, other wrong": r["b"], "DistilBERT wrong, other right": r["c"],
                  "Chi-square": f"{r['chi2']:.2f}", "p": U.fmt_p(r["p"])})
table10b = pd.DataFrame(t); U.save_table(table10b, "table10b_rq2_mcnemar"); table10b

### Figure 11 — Accuracy in the source domain and after the domain shift

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4)); x = np.arange(len(shift)); w = 0.38
ax.bar(x - w / 2, 100 * shift.imdb_acc, w, label="IMDb test (in-domain)", color="#4c566a")
ax.bar(x + w / 2, 100 * shift.amz_acc, w, label="Amazon test (no retraining)", color="#e09f3e")
for i, r in shift.iterrows():
    ax.annotate(f"−{r.drop_pp:.1f} pp", (i + w / 2, 100 * r.amz_acc), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([NAMES[k] for k in shift.model]); ax.set_ylabel("Accuracy (%)")
ax.set_ylim(max(0, 100 * shift.amz_acc.min() - 15), 100); ax.legend(loc="lower right")
ax.set_title("Figure 11. Domain shift: IMDb-trained models on Amazon Electronics")
U.save_fig(fig, "fig11_domain_shift"); plt.show()

## 6. Learning curves on the Amazon target domain
For each size *n* and seed, a balanced random subset of the Amazon fine-tuning pool is drawn. All three conditions share the **same subset**, so differences at each point are paired. The best epoch is chosen on 1,000 Amazon validation reviews, and the score is reported on the 10,000-review Amazon test set.

In [ ]:
pool = T["amz_pool"].reset_index(drop=True)
val_pos = np.sort(np.random.RandomState(0).choice(len(T["amz_val"]), size=min(LC_VAL_N, len(T["amz_val"])), replace=False))
ENC_VAL = [ENC["amz_val"][i] for i in val_pos]; Y_VAL = Y["amz_val"][val_pos]
TXT_VAL = [TXT["amz_val"][i] for i in val_pos]
SRC_PATH = f"{MOD2}/distilbert_imdb_s{SOURCE_SEEDS[0]}"

def subset(n, seed):    # balanced sample of n reviews from the fine-tuning pool
    rs = np.random.RandomState(seed)
    pos = np.where(pool.label.to_numpy() == 1)[0]; neg = np.where(pool.label.to_numpy() == 0)[0]
    return np.sort(np.concatenate([rs.choice(pos, n // 2, replace=False), rs.choice(neg, n - n // 2, replace=False)]))

def run_lr(idx):
    texts = [TXT["amz_pool"][i] for i in idx]; ys = Y["amz_pool"][idx]
    with U.Timer() as t:
        v = TfidfVectorizer(tokenizer=U.tokenize, token_pattern=None, lowercase=False, dtype=np.float32,
                            ngram_range=(1, 2), min_df=2 if len(idx) < 2000 else 3, max_features=50000, sublinear_tf=True)
        X = v.fit_transform(texts); Xv = v.transform(TXT_VAL)
        best = max((LogisticRegression(C=c, solver="liblinear", max_iter=2000).fit(X, ys) for c in C_GRID),
                   key=lambda m: m.score(Xv, Y_VAL))
    return best.predict_proba(v.transform(TXT["amz_test"]))[:, 1], t.seconds / 60

for n in LC_SIZES:
    for seed in LC_SEEDS:
        idx = subset(n, seed)
        for cond in ["distilbert_imdb_adapted", "distilbert_generic", "logreg_scratch"]:
            key = f"{cond}_n{n}_s{seed}"
            if key in done(LC_CSV):
                continue
            print(f"{key} ...")
            if cond == "logreg_scratch":
                p, minutes = run_lr(idx); epochs = np.nan
            else:
                model = new_bert(SRC_PATH if cond == "distilbert_imdb_adapted" else CKPT)
                model, hist, minutes = bert_train(model, [ENC["amz_pool"][i] for i in idx], Y["amz_pool"][idx], ENC_VAL, Y_VAL,
                                                  LC_EPOCHS[n], LC["lr"], LC["batch_size"], LC["warmup"], LC["weight_decay"], seed, verbose=False)
                p = bert_proba(model, ENC["amz_test"]); epochs = int(np.argmax([h["val_acc"] for h in hist]) + 1)
                if cond == "distilbert_imdb_adapted" and n == max(LC_SIZES) and seed == 42:
                    model.save_pretrained(f"{MOD2}/distilbert_amazon_adapted_n{n}_s42"); tok.save_pretrained(f"{MOD2}/distilbert_amazon_adapted_n{n}_s42")
                    np.save(f"{RES}/rq2_adapted_n{n}_s42_amz_test_probs.npy", p)
                del model; torch.cuda.empty_cache()
            m = U.classification_metrics(Y["amz_test"], p)
            U.append_csv(dict(key=key, condition=cond, n=n, seed=seed, accuracy=m["accuracy"], macro_f1=m["macro_f1"],
                              roc_auc=m["roc_auc"], train_time_min=minutes, best_epoch=epochs), LC_CSV)
            print(f"   accuracy {m['accuracy']:.4f} ({minutes:.1f} min)")

### Table 11 and Figure 12 — Learning curves and labels needed

In [ ]:
lc = pd.read_csv(LC_CSV)
# zero-shot points (n = 0): the IMDb models applied directly
zs = pd.DataFrame([dict(condition="distilbert_imdb_adapted", n=0, seed=s, accuracy=shift.set_index("model").loc["distilbert", "amz_acc"]) for s in LC_SEEDS] +
                  [dict(condition="logreg_scratch", n=0, seed=s, accuracy=shift.set_index("model").loc["logreg", "amz_acc"]) for s in LC_SEEDS])
agg = (pd.concat([zs, lc]).groupby(["condition", "n"]).accuracy.agg(["mean", "std", "count"]).reset_index())

IN_DOMAIN = shift.set_index("model").loc["distilbert", "imdb_acc"]
TARGET_CEILING = agg[(agg.condition == "distilbert_imdb_adapted") & (agg.n == max(LC_SIZES))]["mean"].iloc[0]
LABEL = {"distilbert_imdb_adapted": "DistilBERT: IMDb → Amazon", "distilbert_generic": "DistilBERT: generic → Amazon",
         "logreg_scratch": "TF-IDF + LogReg from scratch (IMDb model at n = 0)"}

def labels_needed(cond, ref):
    a = agg[(agg.condition == cond) & (agg.n > 0)].sort_values("n")
    hit = a[a["mean"] >= ref - RECOVERY_PP]
    return int(hit.n.iloc[0]) if len(hit) else None

need = {c: labels_needed(c, IN_DOMAIN) for c in LABEL}
need_ceiling = {c: labels_needed(c, TARGET_CEILING) for c in LABEL}

pivot = agg.pivot(index="n", columns="condition", values="mean"); sd = agg.pivot(index="n", columns="condition", values="std")
table11 = pd.DataFrame(index=pivot.index)
for c in LABEL:
    if c in pivot:
        table11[LABEL[c]] = [("" if pd.isna(m) else f"{100 * m:.2f}%" + ("" if pd.isna(s) or s == 0 else f" ± {100 * s:.2f}")) for m, s in zip(pivot[c], sd[c])]
table11.index.name = "Labeled Amazon reviews (n)"
U.save_table(table11, "table11_rq2_learning_curves", index=True)
print(f"In-domain reference (DistilBERT on IMDb test): {100 * IN_DOMAIN:.2f}%  -> target {100 * (IN_DOMAIN - RECOVERY_PP):.2f}%")
for c in LABEL:
    print(f"  {LABEL[c]:55s} labels to reach within 2 pp of in-domain: {need[c] or 'not reached'}; "
          f"within 2 pp of the n = {max(LC_SIZES)} ceiling: {need_ceiling[c] or 'not reached'}")
json.dump(dict(in_domain_acc=IN_DOMAIN, target_ceiling=TARGET_CEILING, labels_needed_in_domain=need, labels_needed_ceiling=need_ceiling),
          open(f"{RES}/rq2_decision.json", "w"), indent=2)
table11

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
colors = {"distilbert_imdb_adapted": "#2e933c", "distilbert_generic": "#7fb069", "logreg_scratch": "#4c566a"}
for c in LABEL:
    a = agg[agg.condition == c].sort_values("n")
    if a.empty: continue
    xs = a.n.replace(0, LC_SIZES[0] / 2.5)          # draw n = 0 left of the first size on the log axis
    ax.plot(xs, 100 * a["mean"], "o-", color=colors[c], label=LABEL[c])
    ax.fill_between(xs, 100 * (a["mean"] - a["std"].fillna(0)), 100 * (a["mean"] + a["std"].fillna(0)), color=colors[c], alpha=0.15)
ax.axhline(100 * IN_DOMAIN, ls="--", color="k", lw=1, label=f"In-domain IMDb accuracy ({100 * IN_DOMAIN:.1f}%)")
ax.axhspan(100 * (IN_DOMAIN - RECOVERY_PP), 100 * IN_DOMAIN, color="0.85", alpha=0.6, label="Within 2 pp of in-domain")
ax.set_xscale("log"); ax.set_xticks([LC_SIZES[0] / 2.5] + LC_SIZES); ax.set_xticklabels(["0"] + [f"{s:,}" for s in LC_SIZES])
ax.set_xlabel("Labeled Amazon reviews used for training (log scale)"); ax.set_ylabel("Amazon test accuracy (%)")
ax.set_title("Figure 12. Learning curves on the Amazon target domain (mean ± SD over seeds)")
ax.legend(fontsize=8, loc="lower right"); U.save_fig(fig, "fig12_learning_curves"); plt.show()

### Paired comparison at each size: IMDb-pretrained versus generic DistilBERT

In [ ]:
from scipy import stats
pair = lc[lc.condition.isin(["distilbert_imdb_adapted", "distilbert_generic"])].pivot_table(index=["n", "seed"], columns="condition", values="accuracy").dropna()
rows = []
for n, g in pair.groupby(level=0):
    d = g["distilbert_imdb_adapted"] - g["distilbert_generic"]
    p = stats.ttest_rel(g["distilbert_imdb_adapted"], g["distilbert_generic"]).pvalue if len(g) > 1 else np.nan
    rows.append({"n": n, "Mean gain from IMDb pre-adaptation (pp)": f"{100 * d.mean():+.2f}", "SD": f"{100 * d.std(ddof=1):.2f}" if len(g) > 1 else "",
                 "Paired t-test p": U.fmt_p(p) if not np.isnan(p) else "", "Seeds": len(g)})
table11b = pd.DataFrame(rows); U.save_table(table11b, "table11b_rq2_source_gain"); table11b

## Summary for the report
* **Table 10 / Figure 11:** the accuracy drop when movie-review models are applied to product reviews, with McNemar tests (Table 10b) for fine-tuned versus from-scratch models.
* **Table 11 / Figure 12:** learning curves, and the number of labeled target reviews needed to come within 2 pp of in-domain accuracy (`rq2_decision.json`). This is the **labeling guideline** for the transfer playbook.
* **Table 11b:** whether starting from the movie-review model helps beyond generic pre-training.
* The adapted model (`distilbert_amazon_adapted_n5000_s42`) and the IMDb model are saved for RQ4.

**Next:** `05_RQ3_Anomaly_Detection.ipynb`.